-------------------------------------------------------------------------------------------------------
# EUSS Post-Retrofit Measure Packages: MP8, MP9, MP10
-------------------------------------------------------------------------------------------------------
- MP8: Whole Home Electrification (High Efficiency)
- MP9: Whole-Home Electrification + Basic Enclosure Upgrade
- MP10: Whole-Home Electrification + Enhanced Enclosure Upgrade

-------------------------------------------------------------------------------------------------------
# TARE MODEL SCENARIOS
-------------------------------------------------------------------------------------------------------
- Pre-IRA Scenario:
    - NREL End-Use Savings Shapes Database: Measure Package 8/9/10
    - AEO2023 No Inflation Reduction Act
    - Cambium 2021 MidCase
      
- IRA-Reference Scenario:
    - NREL End-Use Savings Shapes Database: Measure Package 8/9/10
    - AEO2023 REFERENCE CASE - HDD and Fuel Price Projections
    - Cambium 2022 and 2023 MidCase

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
import os
from IPython import get_ipython
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Project configuration
from config import PROJECT_ROOT

# Model constants - explicit imports for clarity
from cmu_tare_model.constants import (
    VERBOSE, 
    RCM_MODELS, 
    CR_FUNCTIONS,
    REMDB_COST_SCENARIO_KEYS,
    PRINT_DEBUG,
    PRINT_VERBOSE_DATAFRAMES
)
from cmu_tare_model.constants import (
    PRIVATE_DISCOUNT_RATE_COLS, 
    PRIVATE_DISCOUNT_RATE_SHORT_KEYS
)

# Column name builders
from cmu_tare_model.utils.column_names import (
    create_cost_col,
    create_capital_col,
    create_npv_col,
    create_rebate_col,
    create_adoption_col,
    create_total_npv_col,
    create_health_npv_col,
    create_climate_npv_col
)

# Data loading utility
from cmu_tare_model.utils.load_exported_results_to_df import load_model_run_output, load_measure_package_data

# =============================================================================
# MATPLOTLIB/SEABORN CONFIGURATION
# =============================================================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'
plt.close('all')
%matplotlib inline

sns.set_theme(font='sans-serif', style='darkgrid')

# =============================================================================
# PROJECT ROOT AND TIMESTAMP SETUP
# =============================================================================
# Get the current datetime
start_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Format the name of the exported results file using the location ID
result_export_time = datetime.now()
model_run_date_time = result_export_time.strftime("%Y-%m-%d_%H-%M")

print(f"""
PROJECT_ROOT: {PROJECT_ROOT}

Start Time: {start_time}
Model Run Timestamp: {model_run_date_time}

Active Capital Cost Scenarios: {REMDB_COST_SCENARIO_KEYS}
Note: DataFrames contain columns for ALL active cost scenarios.
Visualizations default to 'v3' with comparative sections for other scenarios.

""")

In [ ]:
# Select whether to begin new run or visualize existing model outputs
while True:
    try:
        start_new_model_run = str(input("""
Would you like to begin a new simulation or visualize output results from a previous model run? Please enter one of the following:
Y. I'd like to start a new model run.
N. I'd like to visualize output results from a previous model run.""")).upper()

        print(f"Enter the following input: {start_new_model_run}")

        if start_new_model_run == 'Y':
            print(f"Formatted date for use in file name: {model_run_date_time}")

            # Relative path to the file from the project root
            relative_path = os.path.join("cmu_tare_model", "model_scenarios", "tare_run_simulation_v2_2.ipynb")

            # Construct the absolute path to the file
            file_path = os.path.join(PROJECT_ROOT, relative_path)
            print(f"File path: {file_path}")

            # Storing Result Outputs in output_results folder
            output_folder_path = os.path.join(PROJECT_ROOT, "cmu_tare_model", "output_results")
            print(f"Result outputs will be exported here: {output_folder_path}")

            # On Windows, to avoid any path-escape quirks, convert backslashes to forward slashes
            file_path = file_path.replace("\\", "/")

            print(f"Running file: {file_path}")

            # iPthon magic command to run a .py file and import variables into the current IPython session
            if os.path.exists(file_path):
                get_ipython().run_line_magic('run', f'-i {file_path}')  # If your path has NO spaces, no quotes needed.
            else:
                print(f"File not found: {file_path}")

            break  # Exit the loop if input is 'Y'
            
        elif start_new_model_run == 'N':
            # Enter the date time of the model run in the following format: YYYY-MM-DD_HH-MM
            model_run_date_time = str(input("Enter the date time of the model run in the following format YYYY-MM-DD_HH-MM: "))
            location_id = str(input("Enter the location ID used in the model run (e.g., 'National' or 'PA'): "))
            
            # Load model run results
            print(f"Loading model run results for location ID: {location_id} and timestamp: {model_run_date_time}")

            # Storing Result Outputs in output_results folder
            output_folder_path = os.path.join(PROJECT_ROOT, "cmu_tare_model", "output_results")
            print(f"Past model run results will be loaded from here: {output_folder_path}")
            
            break  # Exit the loop if input is 'N'
        
        else:
            print("Invalid input. Please enter 'Y' or 'N'.")
    
    except Exception as e:
        print("An error occurred:", e)
        print("Please try again.")

In [ ]:
if VERBOSE:
    print(f"""
    ====================================================================================================================================================================
    LOAD SCENARIO DATA
    ====================================================================================================================================================================
    The load_model_run_output function loads scenario data from a specified folder and date. Additional details are provided below:
        
    Documentation for the load_model_run_output function:
    {load_model_run_output.__doc__}

    -----------------------------------------------------------------------------------------------
    LOADING SCENARIO DATA ...

    These parameters are common to all function calls:
    Output folder path: {output_folder_path}
    Model run date time: {model_run_date_time}
    """)

-------------------------------------------------------------------------------------------------------
# Baseline Scenario: Measure Package 0 (MP0)
-------------------------------------------------------------------------------------------------------

In [ ]:
# =======================================================================================================
# Baseline Scenario: Measure Package 0 (MP0)
# =======================================================================================================
menu_mp = 0

df_outputs_baseline_home = load_model_run_output(
    results_category='summary_baseline',
    menu_mp=menu_mp,
    output_folder_path=output_folder_path,
    location_id=location_id,
    results_export_formatted_date=model_run_date_time,
    use_chunked_loading=True,
    chunk_size=10000
)

-------------------------------------------------------------------------------------------------------
# Basic Retrofit: Measure Package 8 (MP8)
-------------------------------------------------------------------------------------------------------

In [ ]:
# =============================================================================
# LOAD MODEL RESULTS: MP8, MP9, MP10
# =============================================================================
# Load all measure packages
DATAFRAMES_MP8 = load_measure_package_data(8, output_folder_path, location_id, model_run_date_time)
DATAFRAMES_MP9 = load_measure_package_data(9, output_folder_path, location_id, model_run_date_time)
DATAFRAMES_MP10 = load_measure_package_data(10, output_folder_path, location_id, model_run_date_time)

# Convenience mapping for downstream code
DATAFRAMES_BY_MP = {
    8: DATAFRAMES_MP8,
    9: DATAFRAMES_MP9,
    10: DATAFRAMES_MP10
}

# =============================================================================
# DATAFRAME EXTRACTION FOR VISUALIZATIONS
# =============================================================================
# Direct dictionary access with short keys - no helper function needed
# Structure: DATAFRAMES[discount_rate][rcm_model]

# (Using FIXED_BASE as the primary discount rate for most visualizations)
df_outputs_mp8_ap2_FIXED_BASE = DATAFRAMES_MP8['fixed_base']['ap2']
df_outputs_mp8_easiur_FIXED_BASE = DATAFRAMES_MP8['fixed_base']['easiur']
df_outputs_mp8_inmap_FIXED_BASE = DATAFRAMES_MP8['fixed_base']['inmap']

# Sensitivity Analyses DataFrames for MP8
df_outputs_mp8_inmap_FIXED_LOW = DATAFRAMES_MP8['fixed_low']['inmap']
df_outputs_mp8_inmap_FIXED_HIGH = DATAFRAMES_MP8['fixed_high']['inmap']
df_outputs_mp8_inmap_VARIABLE = DATAFRAMES_MP8['variable']['inmap']

# Dataframes used for MP9 and MP10 ASHP comparisons
df_outputs_mp9_ap2_FIXED_BASE = DATAFRAMES_MP9['fixed_base']['ap2']
df_outputs_mp9_easiur_FIXED_BASE = DATAFRAMES_MP9['fixed_base']['easiur']
df_outputs_mp9_inmap_FIXED_BASE = DATAFRAMES_MP9['fixed_base']['inmap']

df_outputs_mp10_ap2_FIXED_BASE = DATAFRAMES_MP10['fixed_base']['ap2']
df_outputs_mp10_easiur_FIXED_BASE = DATAFRAMES_MP10['fixed_base']['easiur']
df_outputs_mp10_inmap_FIXED_BASE = DATAFRAMES_MP10['fixed_base']['inmap']

# CLIMATE CHANGE AND PUBLIC HEALTH IMPACTS

In [ ]:
from cmu_tare_model.utils.data_visualization import print_summary_stats
from cmu_tare_model.utils.data_visualization_boxplots import create_subplot_grid_boxplot
from cmu_tare_model.utils.data_visualization_histograms import create_subplot_grid_histogram, print_positive_percentages_complete

if VERBOSE:
    print(f"""  
    ====================================================================================================================================================================
    UNCERTAINTY ANALYSIS VISUALIZATION
    ====================================================================================================================================================================

    --------------------------------------------------------
    SUMMARY STATISTICS TABLE
    --------------------------------------------------------
    data_visualization.py file contains the documentation for the print_summary_stats function.

    --------------------------------------------------------
    SUBPLOT GRID OF BOXPLOTS
    --------------------------------------------------------
    data_visualization_boxplots.py file contains the documentation for the create_subplot_grid_boxplot function.
        
    --------------------------------------------------------
    SUBPLOT GRID OF HISTOGRAMS
    --------------------------------------------------------
    data_visualization_histograms.py file contains the documentation for the create_subplot_grid_histogram function.
        
    --------------------------------------------------------------------------------------------------------------------------------------------------------------------
    """)

## HEALTH IMPACT: 3 Reduced Complexity Models x 2 CR Functions

In [ ]:
# =============================================================================
# HEALTH IMPACT VISUALIZATIONS: RCM × CR-Function Sensitivity
# =============================================================================
scenario_prefix = 'iraRef_mp8_'
category = 'heating'
lower_percentile = 0.5
upper_percentile = 99.5

# Store figures for later reference
health_npv_figures = {}

for cr_function in CR_FUNCTIONS:  # ['acs', 'h6c']
    print(f"\n{'='*60}")
    print(f"FIGURE: MONETIZED HEALTH IMPACT ({cr_function.upper()} CR-FUNCTION)")
    print(f"{'='*60}")
    
    fig = create_subplot_grid_boxplot(
        dataframes=[
            df_outputs_mp8_ap2_FIXED_BASE,
            df_outputs_mp8_easiur_FIXED_BASE,
            df_outputs_mp8_inmap_FIXED_BASE
        ],
        subplot_positions=[(0, 0), (0, 1), (0, 2)],
        y_cols=[
            f'{scenario_prefix}{category}_health_npv_ap2_{cr_function}',
            f'{scenario_prefix}{category}_health_npv_easiur_{cr_function}',
            f'{scenario_prefix}{category}_health_npv_inmap_{cr_function}'
        ],
        hue_col=f'base_{category}_fuel',
        sharex=True,
        sharey=True,
        subplot_titles=[
            f'AP2 ({cr_function.upper()})', 
            f'EASIUR ({cr_function.upper()})', 
            f'InMAP ({cr_function.upper()})'
        ],
        x_labels=['', '', ''],
        y_labels=['Health NPV [2023 $USD]', '', ''],
        lower_percentile=lower_percentile,
        upper_percentile=upper_percentile,
        figure_size=(16, 6),
        show_outliers=False,
        show_xtick_labels=False
    )

    # Print summary statistics tables
    # ===== AP2 =====
    print_summary_stats(dataframes=[df_outputs_mp8_ap2_FIXED_BASE],
                        column_names=[f'{scenario_prefix}{category}_health_npv_ap2_{cr_function}'],
                        subplot_titles=[f'AP2 with {cr_function.upper()} CR-Function'])
    # ===== EASIUR =====
    print_summary_stats(dataframes=[df_outputs_mp8_easiur_FIXED_BASE],
                        column_names=[f'{scenario_prefix}{category}_health_npv_easiur_{cr_function}'],
                        subplot_titles=[f'EASIUR with {cr_function.upper()} CR-Function'])
    # ===== InMAP =====
    print_summary_stats(dataframes=[df_outputs_mp8_inmap_FIXED_BASE],
                        column_names=[f'{scenario_prefix}{category}_health_npv_inmap_{cr_function}'],
                        subplot_titles=[f'InMAP with {cr_function.upper()} CR-Function'])

    # Print positive percentage statistics
    print_positive_percentages_complete(
        dataframes=[
            df_outputs_mp8_ap2_FIXED_BASE,
            df_outputs_mp8_easiur_FIXED_BASE,
            df_outputs_mp8_inmap_FIXED_BASE
        ],
        column_names=[
            f'{scenario_prefix}{category}_health_npv_ap2_{cr_function}',
            f'{scenario_prefix}{category}_health_npv_easiur_{cr_function}',
            f'{scenario_prefix}{category}_health_npv_inmap_{cr_function}'
        ],
        subplot_titles=[
            f'AP2 ({cr_function.upper()})',
            f'EASIUR ({cr_function.upper()})',
            f'InMAP ({cr_function.upper()})'
        ],
        fuel_column=f'base_{category}_fuel'
    )

    # Store figure
    health_npv_figures[cr_function] = fig
    
    # Display
    display(fig)

## Climate Change Impact (SCC) and Tier 3 Adopters

### Space Heating - Progressive Impact of Climate Benefit Valuation

In [ ]:
scenario_prefix = 'iraRef_mp8_'
category = 'heating'
discount_rate = 'fixed_base'
cost_scenario = 'v4MID'  # Default cost scenario for visualization
lower_percentile = 0.5
upper_percentile = 99.5

# Build column names using centralized builders
private_npv_col = create_npv_col(scenario_prefix, category, 'moreWTP', cost_scenario, f'_{discount_rate}')
climate_npv_lower = create_total_npv_col(scenario_prefix, category, cost_scenario=cost_scenario,
                                          method_suffix=f'_{discount_rate}', scc_assumption='lower', climate_only=True)
climate_npv_central = create_total_npv_col(scenario_prefix, category, cost_scenario=cost_scenario,
                                            method_suffix=f'_{discount_rate}', scc_assumption='central', climate_only=True)
climate_npv_upper = create_total_npv_col(scenario_prefix, category, cost_scenario=cost_scenario,
                                          method_suffix=f'_{discount_rate}', scc_assumption='upper', climate_only=True)

print(f"""
===== FIGURE 7: CLIMATE BENEFIT IMPACT ON RETROFIT ADOPTION POTENTIAL (TIER 3) =====
- Retrofit Scenarios: {scenario_prefix} 
- Discount Rate: {discount_rate}
- Cost Scenario: {cost_scenario}
- Categories: {category}

Valid Range: {lower_percentile}th to {upper_percentile}th Percentile

Column names:
  Private NPV: {private_npv_col}
  Climate Lower: {climate_npv_lower}
  Climate Central: {climate_npv_central}
  Climate Upper: {climate_npv_upper}
""")

fig_heating_climate_scc_FIXED_BASE = create_subplot_grid_histogram(
    dataframes=[
        df_outputs_mp8_inmap_FIXED_BASE,
        df_outputs_mp8_inmap_FIXED_BASE,
        df_outputs_mp8_inmap_FIXED_BASE,
        df_outputs_mp8_inmap_FIXED_BASE
        ],
    subplot_positions=[(0, 0), (0, 1), (0, 2), (0, 3)],  # 1x4 grid
    x_cols=[
        private_npv_col,
        climate_npv_lower,
        climate_npv_central,
        climate_npv_upper
    ],
    x_labels=['Private NPV [2023 $USD]'] + ['Total NPV [2023 $USD]'] * 3,
    y_labels=['Dwelling Units', '', '', ''],
    bin_number=40,  # Optional: number of bins for histogram
    lower_percentile=lower_percentile,    # Show nearly full range
    upper_percentile=upper_percentile,   # Show nearly full range
    subplot_titles=[
        'Private NPV Only\n37% Positive NPV',
        'SCC Lower Bound\n56% Positive NPV',
        'SCC Central Estimate\n78% Positive NPV',
        'SCC Upper Bound\n83% Positive NPV' 
    ],
    # suptitle=f'{category.title()}: Progressive Impact of Climate Benefit Valuation',
    figure_size=(20, 10),  # Wide format for 4 panels
    sharex=False,  # Keep different scales to show full distributions
    sharey=True,   # Same y-scale for comparison
    color_code=f'base_{category}_fuel'
)

print_positive_percentages_complete(
    df=df_outputs_mp8_inmap_FIXED_BASE,
    column_names=[
        private_npv_col,
        climate_npv_lower,
        climate_npv_central,
        climate_npv_upper
    ],
    subplot_titles=[
        f'Private NPV Only (Baseline), Discount Rate: {discount_rate}', 
        f'Lower Bound SCC (+ Climate), Discount Rate: {discount_rate}', 
        f'Central Estimate SCC (+ Climate), Discount Rate: {discount_rate}', 
        f'Upper Bound SCC (+ Climate), Discount Rate: {discount_rate}'
    ],
    fuel_column=f'base_{category}_fuel'
)

fig_heating_climate_scc_FIXED_BASE

# Adoption Rate Scenario Comparison

In [ ]:
from cmu_tare_model.adoption_potential.data_processing.visuals_adoption_potential import (
    create_multiIndex_adoption_df,
    print_adoption_decision_percentages,
    subplot_grid_adoption_vBar
)

if VERBOSE:

    print(f"""  
    ====================================================================================================================================================================
    ADOPTION POTENTIAL VISUALIZATION
    ====================================================================================================================================================================

    --------------------------------------------------------
    CREATE MULTI-INDEX DF FOR ADOPTION POTENTIAL
    --------------------------------------------------------
    visuals_adoption_potential.py file contains the documentation for the create_multiIndex_adoption_df function.

    --------------------------------------------------------
    VISUALIZE ADOPTION POTENTIAL SUBPLOT GRID
    --------------------------------------------------------
    visuals_adoption_potential.py file contains the documentation for the subplot_grid_adoption_vBar function.
        
    --------------------------------------------------------------------------------------------------------------------------------------------------------------------

    """)

## Space Heating - Basic (MP8), Moderate (MP9), Advanced (MP10) Retrofit


In [ ]:
# =============================================================================
# CREATE ADOPTION POTENTIAL DATAFRAMES (ALL COMBINATIONS)
# =============================================================================
# Creates DataFrames for all sensitivity combinations upfront.
# Structure: ALL_HEATING_ADOPTION_MI[mp][cost_scenario][discount_rate][rcm][crf] = DataFrame
# The cost_scenario dimension is new in v2.3.

scc = 'central'
HEATING_MEASURE_PACKAGES = [8, 9, 10]

# Master dictionary to store all results
# Structure: [mp][cost_scenario][discount_rate][rcm][crf]
ALL_HEATING_ADOPTION_MI = {
    mp: {
        cs: {
            discount_rate: {
                rcm: {crf: None for crf in CR_FUNCTIONS} 
                for rcm in RCM_MODELS
            }
            for discount_rate in PRIVATE_DISCOUNT_RATE_SHORT_KEYS
        }
        for cs in REMDB_COST_SCENARIO_KEYS
    }
    for mp in HEATING_MEASURE_PACKAGES
}

print("Creating adoption potential DataFrames...")
print(f"Cost scenarios: {REMDB_COST_SCENARIO_KEYS}")

for menu_mp in HEATING_MEASURE_PACKAGES:
    print(f"\n{'='*80}")
    print(f"MEASURE PACKAGE {menu_mp}")
    print(f"{'='*80}")

    for cost_scenario in REMDB_COST_SCENARIO_KEYS:
        print(f"\n  Cost Scenario: {cost_scenario}")

        for discount_rate in PRIVATE_DISCOUNT_RATE_SHORT_KEYS:
            print(f"    Discount Rate: {discount_rate}")

            for rcm_model in RCM_MODELS:
                for cr_function in CR_FUNCTIONS:
                    # Direct dictionary access with short keys
                    source_df = DATAFRAMES_BY_MP[menu_mp][discount_rate][rcm_model]
                    
                    df_mi = create_multiIndex_adoption_df(
                        df=source_df,
                        menu_mp=menu_mp,
                        category='heating',
                        scc=scc,
                        rcm_model=rcm_model,
                        cr_function=cr_function,
                        cost_scenario=cost_scenario,
                        discount_rate=discount_rate
                    )
                    
                    ALL_HEATING_ADOPTION_MI[menu_mp][cost_scenario][discount_rate][rcm_model][cr_function] = df_mi

total_dfs = (len(REMDB_COST_SCENARIO_KEYS) * len(PRIVATE_DISCOUNT_RATE_SHORT_KEYS) * 
             len(HEATING_MEASURE_PACKAGES) * len(RCM_MODELS) * len(CR_FUNCTIONS))
print(f"\n✓ Created {total_dfs} DataFrames ({len(REMDB_COST_SCENARIO_KEYS)} cost scenarios × "
      f"{len(PRIVATE_DISCOUNT_RATE_SHORT_KEYS)} discount rates × {len(HEATING_MEASURE_PACKAGES)} MPs × "
      f"{len(RCM_MODELS)} RCMs × {len(CR_FUNCTIONS)} CRFs)")

In [ ]:
# =============================================================================
# VISUALIZATION CONFIGURATION
# =============================================================================
# Edit these values, then run the next cell to create the visualization.

# Discount rate: 'fixed_low', 'fixed_base', 'fixed_high', 'variable'
discount_rate = 'fixed_base'

# Capital cost scenario: 'v3', 'v4MID', etc.
cost_scenario = 'v4MID'

# Health model parameters (typically keep these fixed)
scc = 'central'
rcm_model = 'inmap'
cr_function = 'acs'

# =============================================================================
# ADOPTION POTENTIAL VISUALIZATION
# =============================================================================
# Subplot titles and labels for each measure package
MP_SUBTITLES = {
    8: "ASHP Only:\nNo IRA vs. IRA-Reference",
    9: "ASHP + Basic Enclosure:\nNo IRA vs. IRA-Reference",
    10: "ASHP + Enhanced Enclosure:\nNo IRA vs. IRA-Reference"
}

print(f"""
================================================================================
ADOPTION POTENTIAL VISUALIZATION
================================================================================
Discount Rate: {discount_rate}
Cost Scenario: {cost_scenario}
SCC: {scc} | RCM: {rcm_model} | CRF: {cr_function}
""")

# Build adoption column names with cost_scenario included
def build_adoption_scenario_names(mp, scc, rcm, crf, cs, dr):
    """Build preIRA and iraRef adoption column names for a given MP."""
    return [
        f'preIRA_mp{mp}_heating_adoption_{scc}_{rcm}_{crf}_{cs}_{dr}',
        f'iraRef_mp{mp}_heating_adoption_{scc}_{rcm}_{crf}_{cs}_{dr}'
    ]

fig_adoption = subplot_grid_adoption_vBar(
    dataframes=[
        ALL_HEATING_ADOPTION_MI[8][cost_scenario][discount_rate][rcm_model][cr_function],
        ALL_HEATING_ADOPTION_MI[9][cost_scenario][discount_rate][rcm_model][cr_function], 
        ALL_HEATING_ADOPTION_MI[10][cost_scenario][discount_rate][rcm_model][cr_function]
    ],
    scenarios_list=[
        build_adoption_scenario_names(8, scc, rcm_model, cr_function, cost_scenario, discount_rate),
        build_adoption_scenario_names(9, scc, rcm_model, cr_function, cost_scenario, discount_rate),
        build_adoption_scenario_names(10, scc, rcm_model, cr_function, cost_scenario, discount_rate)
    ],
    subplot_positions=[(0, 0), (0, 1), (0, 2)],
    filter_fuel=['Electricity', 'Natural Gas', 'Fuel Oil', 'Propane'],
    x_labels=["", "Fuel Type and Income Group (LMI: Low-to-Moderate-Income, MUI: Middle-to-Upper-Income)", ""],
    plot_titles=[MP_SUBTITLES[mp] for mp in HEATING_MEASURE_PACKAGES],
    y_labels=["Retrofit Adoption Potential (%)", "", ""],
    # suptitle=f"Space Heating Air-Source Heat Pump (ASHP) Retrofit Scenario Comparison\nClimate Sensitivity: SCC-{scc.upper()} | Health Sensitivity: {rcm_model.upper()}-{cr_function.upper()}",
    figure_size=(18, 12),
    sharey=True,
    x_tick_format="all"  # Use LMI/MUI classification for x-ticks
)

# =======================================================================================================
# PRINT ADOPTION DECISION PERCENTAGES FOR INMAP-ACS, FIXED-BASE DISCOUNT RATE
# =======================================================================================================
for i, menu_mp in enumerate(HEATING_MEASURE_PACKAGES):
    scenario_names = build_adoption_scenario_names(menu_mp, scc, rcm_model, cr_function, cost_scenario, discount_rate)
    print_adoption_decision_percentages(
            dataframes=[
                ALL_HEATING_ADOPTION_MI[menu_mp][cost_scenario][discount_rate][rcm_model][cr_function],
                ALL_HEATING_ADOPTION_MI[menu_mp][cost_scenario][discount_rate][rcm_model][cr_function],
                ],
            scenario_names=scenario_names,
            source_dataframes=[
                DATAFRAMES_BY_MP[menu_mp][discount_rate][rcm_model],
                DATAFRAMES_BY_MP[menu_mp][discount_rate][rcm_model],
            ],
            category='heating',
            title=f"SPACE HEATING ADOPTION POTENTIAL: {discount_rate.upper()} | Cost: {cost_scenario}", 
            subtitle=MP_SUBTITLES[menu_mp],
            print_header_key=True,
        )

fig_adoption

## Water Heating, Clothes Drying, and Cooking - Basic Retrofit (MP8)

In [ ]:
# =============================================================================
# CREATE ADOPTION POTENTIAL DFs FOR NON-HVAC CATEGORIES (ALL DISCOUNT RATES)
# =============================================================================
menu_mp = 8
scc = 'central'
cost_scenario = 'v4MID'  # Default cost scenario
CATEGORIES = ['waterHeating', 'clothesDrying', 'cooking']

# Filter to only categories that are active in EQUIPMENT_SPECS
from cmu_tare_model.constants import VALID_CATEGORIES
ACTIVE_NONHVAC = [c for c in CATEGORIES if c in VALID_CATEGORIES]

if not ACTIVE_NONHVAC:
    print(f"""
=======================================================================================================
BASIC RETROFIT: MEASURE PACKAGE {menu_mp} (MP{menu_mp}) - NON-HVAC CATEGORIES
=======================================================================================================
NOTE: No non-HVAC categories are currently active in EQUIPMENT_SPECS.
Active categories: {VALID_CATEGORIES}
Skipping non-HVAC adoption analysis.
=======================================================================================================
""")
    MP8_NONHVAC_ADOPTION_MI = {}
else:
    print(f"""
=======================================================================================================
BASIC RETROFIT: MEASURE PACKAGE {menu_mp} (MP{menu_mp}) - NON-HVAC CATEGORIES
=======================================================================================================

Creating Multi-Index DataFrames for:
- Categories: {ACTIVE_NONHVAC}
- Cost Scenarios: {REMDB_COST_SCENARIO_KEYS}
- Discount Rates: {PRIVATE_DISCOUNT_RATE_SHORT_KEYS}
- RCM Models: {RCM_MODELS}
- CR Functions: {CR_FUNCTIONS}

""")

    # Initialize nested dictionary to store results
    # Structure: [category][cost_scenario][discount_rate][rcm][crf]
    MP8_NONHVAC_ADOPTION_MI = {
        category: {
            cs: {
                discount_rate: {
                    rcm: {crf: None for crf in CR_FUNCTIONS}
                    for rcm in RCM_MODELS
                }
                for discount_rate in PRIVATE_DISCOUNT_RATE_SHORT_KEYS
            }
            for cs in REMDB_COST_SCENARIO_KEYS
        }
        for category in ACTIVE_NONHVAC
    }

    # Category display names for pretty printing
    CATEGORY_NAMES = {
        'waterHeating': 'Water Heating',
        'clothesDrying': 'Clothes Drying',
        'cooking': 'Cooking'
    }

    # Create all combinations using nested loops
    for category in ACTIVE_NONHVAC:
        print(f"\n{'='*80}")
        print(f"CATEGORY: {CATEGORY_NAMES.get(category, category).upper()}")
        print(f"{'='*80}")
        
        for cs in REMDB_COST_SCENARIO_KEYS:
            print(f"\n  Cost Scenario: {cs}")
            
            for discount_rate_short in PRIVATE_DISCOUNT_RATE_SHORT_KEYS:
                print(f"    Discount Rate: {discount_rate_short}")
                
                for rcm_model in RCM_MODELS:
                    for cr_function in CR_FUNCTIONS:
                        source_df = DATAFRAMES_MP8[discount_rate_short][rcm_model]
                        
                        df_mi = create_multiIndex_adoption_df(
                            df=source_df,
                            menu_mp=menu_mp,
                            category=category,
                            scc=scc,
                            rcm_model=rcm_model,
                            cr_function=cr_function,
                            cost_scenario=cs,
                            discount_rate=discount_rate_short
                        )
                        
                        MP8_NONHVAC_ADOPTION_MI[category][cs][discount_rate_short][rcm_model][cr_function] = df_mi

    print(f"\n{'='*80}")
    print(f"COMPLETE: Created adoption DataFrames for {len(ACTIVE_NONHVAC)} non-HVAC categories")
    print(f"{'='*80}\n")

In [ ]:
# ====================================================================
# VISUALIZATION: Water Heating, Clothes Drying, Cooking - MP8
# ====================================================================
scc = 'central'
rcm_model = 'inmap'
cr_function = 'acs'
discount_rate_short = 'fixed_base'
cost_scenario = 'v4MID'  # Default cost scenario

# Category display names
CATEGORY_NAMES = {
    'waterHeating': 'Water Heating',
    'clothesDrying': 'Clothes Drying',
    'cooking': 'Cooking'
}

if not ACTIVE_NONHVAC:
    print("No active non-HVAC categories. Skipping visualization.")
else:
    print(f"""
    SENSITIVITY:
    - SCC Climate Sensitivity: {scc}
    - Discount Rate: {discount_rate_short}
    - Cost Scenario: {cost_scenario}
    - Health RCM Model: {rcm_model}
    - Health CR Function: {cr_function}
    """)

    # Access using consistent structure: [category][cost_scenario][discount_rate][rcm][crf]
    fig_mp8_nonHVAC = subplot_grid_adoption_vBar(
        dataframes=[
            MP8_NONHVAC_ADOPTION_MI[cat][cost_scenario][discount_rate_short][rcm_model][cr_function]
            for cat in ACTIVE_NONHVAC
        ],
        scenarios_list=[
            [f'preIRA_mp8_{cat}_adoption_{scc}_{rcm_model}_{cr_function}_{cost_scenario}_{discount_rate_short}',
             f'iraRef_mp8_{cat}_adoption_{scc}_{rcm_model}_{cr_function}_{cost_scenario}_{discount_rate_short}']
            for cat in ACTIVE_NONHVAC
        ],
        subplot_positions=[(0, i) for i in range(len(ACTIVE_NONHVAC))],
        filter_fuel=['Electricity', 'Natural Gas', 'Fuel Oil', 'Propane'],
        x_labels=["", "Fuel Type and Income Group (LMI: Low-to-Moderate-Income, MUI: Middle-to-Upper-Income)", ""],
        plot_titles=[
            f"{CATEGORY_NAMES.get(cat, cat)}:\nNo IRA vs. IRA-Reference"
            for cat in ACTIVE_NONHVAC
        ],
        y_labels=["Retrofit Adoption Potential (%)"] + [""] * (len(ACTIVE_NONHVAC) - 1),
        figure_size=(18, 12),
        sharey=True,
        x_tick_format="all"
    )

    # Print statistics
    for category in ACTIVE_NONHVAC:
        print_adoption_decision_percentages(
            dataframes=[
                MP8_NONHVAC_ADOPTION_MI[category][cost_scenario][discount_rate_short][rcm_model][cr_function],
                MP8_NONHVAC_ADOPTION_MI[category][cost_scenario][discount_rate_short][rcm_model][cr_function]
            ],
            scenario_names=[
                f'preIRA_mp8_{category}_adoption_{scc}_{rcm_model}_{cr_function}_{cost_scenario}_{discount_rate_short}',
                f'iraRef_mp8_{category}_adoption_{scc}_{rcm_model}_{cr_function}_{cost_scenario}_{discount_rate_short}'
            ],
            source_dataframes=[
                DATAFRAMES_MP8[discount_rate_short][rcm_model],
                DATAFRAMES_MP8[discount_rate_short][rcm_model]
            ],
            category=category,
            title=f"NON-HVAC ADOPTION: {discount_rate_short.upper()} | Cost: {cost_scenario}",
            subtitle=CATEGORY_NAMES.get(category, category),
            print_header_key=True
        )

    fig_mp8_nonHVAC

# SENSITIVITY ANALYSIS: Private Discount Rate and Adoption Feasibility (Retrofit Lifecycle Cost)

In [ ]:
# Discount Rate Sensitivity Analysis
category = 'heating'
cost_scenario = 'v4MID'  # Default cost scenario

# Build NPV column names using centralized builders
preIRA_npv_cols = {
    dr: create_npv_col(f'preIRA_mp8_', category, 'moreWTP', cost_scenario, f'_{dr}')
    for dr in PRIVATE_DISCOUNT_RATE_SHORT_KEYS
}

fig_HEATING_preIRA_private_more_WTP_discount = create_subplot_grid_histogram(
    dataframes=[
        df_outputs_mp8_inmap_FIXED_LOW, 
        df_outputs_mp8_inmap_FIXED_BASE,
        df_outputs_mp8_inmap_FIXED_HIGH,
        df_outputs_mp8_inmap_VARIABLE
    ],
    subplot_positions=[(0, 0), (0, 1), (0, 2), (0, 3)],
    x_cols=[
        preIRA_npv_cols['fixed_low'],
        preIRA_npv_cols['fixed_base'],
        preIRA_npv_cols['fixed_high'],
        preIRA_npv_cols['variable']
    ],
    x_labels=['Private NPV [$USD2023]',
              'Private NPV [$USD2023]',
              'Private NPV [$USD2023]',
              'Private NPV [$USD2023]'
    ],
    y_labels=['Dwelling units in Pre-IRA Scenario', '', '', ''],
    bin_number='auto',
    lower_percentile=lower_percentile,
    upper_percentile=upper_percentile,
    subplot_titles=['Fixed Discount Rate\n Low (2%)',
                    'Fixed Discount Rate\n Base (7%)',
                    'Fixed Discount Rate\n High (12%)',
                    'Variable Discount Rate\n Inverse to % AMI (7% to 45%)'],
    figure_size=(20, 10),  # Wide format for 4 panels
    sharex=False,  # Keep different scales to show full distributions
    sharey=True,   # Same y-scale for comparison
    color_code=f'base_{category}_fuel',
    show_legend=True
)

# Print comparison statistics
print("="*60)
print("Pre-IRA Scenario\nAdoption Feasibility under Different Discount Rate Assumptions")
print(f"Cost Scenario: {cost_scenario}")
print("="*60)

print_positive_percentages_complete(
    dataframes=[
        df_outputs_mp8_inmap_FIXED_LOW, 
        df_outputs_mp8_inmap_FIXED_BASE,
        df_outputs_mp8_inmap_FIXED_HIGH,
        df_outputs_mp8_inmap_VARIABLE
    ],
    column_names=[
        preIRA_npv_cols['fixed_low'],
        preIRA_npv_cols['fixed_base'],
        preIRA_npv_cols['fixed_high'],
        preIRA_npv_cols['variable']
    ],
    subplot_titles=['Fixed Discount Rate Low (2%)',
                    'Fixed Discount Rate Base (7%)',
                    'Fixed Discount Rate High (12%)',
                    'Variable Discount Rate Inverse to % AMI (7% to 45%)'],
    fuel_column=f'base_{category}_fuel'
)

fig_HEATING_preIRA_private_more_WTP_discount

In [ ]:
# Discount Rate Sensitivity Analysis
category = 'heating'
cost_scenario = 'v4MID'  # Default cost scenario

# Build NPV column names using centralized builders
iraRef_npv_cols = {
    dr: create_npv_col(f'iraRef_mp8_', category, 'moreWTP', cost_scenario, f'_{dr}')
    for dr in PRIVATE_DISCOUNT_RATE_SHORT_KEYS
}

fig_HEATING_iraRef_private_more_WTP_discount = create_subplot_grid_histogram(
    dataframes=[
        df_outputs_mp8_inmap_FIXED_LOW, 
        df_outputs_mp8_inmap_FIXED_BASE,
        df_outputs_mp8_inmap_FIXED_HIGH,
        df_outputs_mp8_inmap_VARIABLE
    ],
    subplot_positions=[(0, 0), (0, 1), (0, 2), (0, 3)],
    x_cols=[
        iraRef_npv_cols['fixed_low'],
        iraRef_npv_cols['fixed_base'],
        iraRef_npv_cols['fixed_high'],
        iraRef_npv_cols['variable']
    ],
    x_labels=['Private NPV [$USD2023]',
              'Private NPV [$USD2023]',
              'Private NPV [$USD2023]',
              'Private NPV [$USD2023]'
    ],
    y_labels=['Dwelling units in IRA-Reference Scenario', '', '', ''],
    bin_number='auto',
    lower_percentile=lower_percentile,
    upper_percentile=upper_percentile,
    subplot_titles=['Fixed Discount Rate\n Low (2%)',
                    'Fixed Discount Rate\n Base (7%)',
                    'Fixed Discount Rate\n High (12%)',
                    'Variable Discount Rate\n Inverse to % AMI (7% to 45%)'],
    figure_size=(20, 10),  # Wide format for 4 panels
    sharex=False,  # Keep different scales to show full distributions
    sharey=True,   # Same y-scale for comparison
    color_code=f'base_{category}_fuel',
    show_legend=True
)

# Print comparison statistics
print("="*60)
print("IRA-Reference Scenario\nAdoption Feasibility under Different Discount Rate Assumptions")
print(f"Cost Scenario: {cost_scenario}")
print("="*60)

print_positive_percentages_complete(
    dataframes=[
        df_outputs_mp8_inmap_FIXED_LOW, 
        df_outputs_mp8_inmap_FIXED_BASE,
        df_outputs_mp8_inmap_FIXED_HIGH,
        df_outputs_mp8_inmap_VARIABLE
    ],
    column_names=[
        iraRef_npv_cols['fixed_low'],
        iraRef_npv_cols['fixed_base'],
        iraRef_npv_cols['fixed_high'],
        iraRef_npv_cols['variable']
    ],
    subplot_titles=['Fixed Discount Rate Low (2%)',
                    'Fixed Discount Rate Base (7%)',
                    'Fixed Discount Rate High (12%)',
                    'Variable Discount Rate Inverse to % AMI (7% to 45%)'],
    fuel_column=f'base_{category}_fuel'
)

fig_HEATING_iraRef_private_more_WTP_discount

## Capital Cost Scenario Sensitivity: Adoption Rate Comparison

Compare adoption rates and NPV distributions across capital cost estimation methods (v3 probabilistic vs v4MID REMDB regression).

In [ ]:
# ====================================================================
# COST SCENARIO SENSITIVITY: Adoption Rate Comparison
# ====================================================================
# Compare adoption rates across cost estimation methods for each MP.
# Default analysis parameters (consistent with earlier sections):
scc = 'central'
rcm_model = 'inmap'
cr_function = 'acs'
discount_rate_short = 'fixed_base'
category = 'heating'

if len(REMDB_COST_SCENARIO_KEYS) < 2:
    print("Only one cost scenario active — skipping comparison.")
else:
    print(f"""
    COST SCENARIO SENSITIVITY ANALYSIS
    ===================================
    Active Cost Scenarios: {REMDB_COST_SCENARIO_KEYS}
    Parameters: SCC={scc}, RCM={rcm_model}, CRF={cr_function}, DR={discount_rate_short}
    """)

    # ─── Per-MP Adoption Bar Charts (side-by-side cost scenarios) ───
    for menu_mp in HEATING_MEASURE_PACKAGES:
        scenario_prefix_pre = f'preIRA_mp{menu_mp}_'
        scenario_prefix_ira = f'iraRef_mp{menu_mp}_'

        # Build scenario names for each cost scenario
        scenario_data = {}
        for cs in REMDB_COST_SCENARIO_KEYS:
            col_pre = create_adoption_col(
                scenario_prefix_pre, category, 'adoption', cs,
                f'_{discount_rate_short}',
                scc_assumption=scc, rcm_model=rcm_model, cr_function=cr_function)
            col_ira = create_adoption_col(
                scenario_prefix_ira, category, 'adoption', cs,
                f'_{discount_rate_short}',
                scc_assumption=scc, rcm_model=rcm_model, cr_function=cr_function)
            scenario_data[cs] = {'preIRA': col_pre, 'iraRef': col_ira}

        # Get the source dataframe (all cost scenario columns are in same df)
        source_df = DATAFRAMES_BY_MP[menu_mp][discount_rate_short][rcm_model]

        # Calculate and print mean adoption rates per cost scenario
        print(f"\n{'='*80}")
        print(f"MP{menu_mp} HEATING — Mean Adoption Rates by Cost Scenario")
        print(f"{'='*80}")
        print(f"{'Cost Scenario':<18} {'Pre-IRA Mean':>15} {'IRA-Ref Mean':>15} {'IRA Uplift':>15}")
        print(f"{'-'*63}")

        for cs in REMDB_COST_SCENARIO_KEYS:
            pre_mean = source_df[scenario_data[cs]['preIRA']].mean() * 100
            ira_mean = source_df[scenario_data[cs]['iraRef']].mean() * 100
            uplift = ira_mean - pre_mean
            print(f"{cs:<18} {pre_mean:>14.2f}% {ira_mean:>14.2f}% {uplift:>+14.2f}%")

    # ─── NPV Distribution Comparison ───
    print(f"\n\n{'='*80}")
    print("NPV DISTRIBUTION COMPARISON ACROSS COST SCENARIOS")
    print(f"{'='*80}")

    fig_npv_cost, axes_npv_cost = plt.subplots(
        len(HEATING_MEASURE_PACKAGES), len(REMDB_COST_SCENARIO_KEYS),
        figsize=(7 * len(REMDB_COST_SCENARIO_KEYS), 5 * len(HEATING_MEASURE_PACKAGES)),
        sharey='row', squeeze=False
    )

    for row, menu_mp in enumerate(HEATING_MEASURE_PACKAGES):
        source_df = DATAFRAMES_BY_MP[menu_mp][discount_rate_short][rcm_model]
        scenario_prefix_ira = f'iraRef_mp{menu_mp}_'

        for col_idx, cs in enumerate(REMDB_COST_SCENARIO_KEYS):
            ax = axes_npv_cost[row, col_idx]

            npv_col_more = create_npv_col(
                scenario_prefix_ira, category, 'moreWTP', cs, f'_{discount_rate_short}')
            npv_col_less = create_npv_col(
                scenario_prefix_ira, category, 'lessWTP', cs, f'_{discount_rate_short}')

            if npv_col_more in source_df.columns:
                data_more = source_df[npv_col_more].dropna()
                data_less = source_df[npv_col_less].dropna()

                ax.hist(data_more, bins=50, alpha=0.6, label='More WTP', color='steelblue', density=True)
                ax.hist(data_less, bins=50, alpha=0.6, label='Less WTP', color='coral', density=True)
                ax.axvline(x=0, color='black', linestyle='--', linewidth=0.8, alpha=0.7)

                ax.set_title(f'MP{menu_mp} — Cost Scenario: {cs}', fontsize=11, fontweight='bold')
                ax.set_xlabel('Private NPV ($)')
                if col_idx == 0:
                    ax.set_ylabel('Density')
                ax.legend(fontsize=8)

                # Add median annotation
                median_more = data_more.median()
                median_less = data_less.median()
                ax.axvline(x=median_more, color='steelblue', linestyle=':', linewidth=1)
                ax.axvline(x=median_less, color='coral', linestyle=':', linewidth=1)
            else:
                ax.text(0.5, 0.5, f'{cs} columns\nnot found',
                        transform=ax.transAxes, ha='center', va='center', fontsize=10)

    fig_npv_cost.suptitle(
        f'Private NPV Distribution by Cost Scenario\n'
        f'(SCC={scc}, RCM={rcm_model}, CRF={cr_function}, DR={discount_rate_short})',
        fontsize=14, fontweight='bold', y=1.02
    )
    fig_npv_cost.tight_layout()
    plt.show()

    # ─── Summary Statistics Table ───
    print(f"\n{'='*80}")
    print("PRIVATE NPV SUMMARY STATISTICS BY COST SCENARIO")
    print(f"{'='*80}")
    print(f"{'MP':<6} {'Cost Scen.':<12} {'WTP':<10} {'Median ($)':>14} {'Mean ($)':>14} {'Std Dev ($)':>14} {'% Positive':>12}")
    print(f"{'-'*82}")

    for menu_mp in HEATING_MEASURE_PACKAGES:
        source_df = DATAFRAMES_BY_MP[menu_mp][discount_rate_short][rcm_model]
        scenario_prefix_ira = f'iraRef_mp{menu_mp}_'

        for cs in REMDB_COST_SCENARIO_KEYS:
            for wtp_label, wtp in [('More WTP', 'moreWTP'), ('Less WTP', 'lessWTP')]:
                npv_col = create_npv_col(
                    scenario_prefix_ira, category, wtp, cs, f'_{discount_rate_short}')
                if npv_col in source_df.columns:
                    data = source_df[npv_col].dropna()
                    pct_pos = (data > 0).sum() / len(data) * 100
                    print(f"MP{menu_mp:<4} {cs:<12} {wtp_label:<10} "
                          f"{data.median():>14,.0f} {data.mean():>14,.0f} "
                          f"{data.std():>14,.0f} {pct_pos:>11.1f}%")
        print()  # blank line between MPs

## Cross-Measure-Package Sensitivity Analysis

Compares adoption rates, capital costs, and NPV distributions across MPs (8, 9, 10) and cost scenarios to assess the relative impact of retrofit depth vs. cost estimation methodology.

In [ ]:
# ====================================================================
# CROSS-MP SENSITIVITY ANALYSIS
# ====================================================================
# Compare MP8 (Basic), MP9 (Moderate), MP10 (Advanced) across cost scenarios.
scc = 'central'
rcm_model = 'inmap'
cr_function = 'acs'
discount_rate_short = 'fixed_base'
category = 'heating'

MP_LABELS = {8: 'MP8 (Basic)', 9: 'MP9 (Moderate)', 10: 'MP10 (Advanced)'}

# ─── 1. GROUPED BAR CHART: Mean Adoption by MP × Cost Scenario ───
print(f"""
CROSS-MP SENSITIVITY ANALYSIS
==============================
Parameters: SCC={scc}, RCM={rcm_model}, CRF={cr_function}, DR={discount_rate_short}
Cost Scenarios: {REMDB_COST_SCENARIO_KEYS}
""")

# Collect data for plotting
adoption_summary = []
for menu_mp in HEATING_MEASURE_PACKAGES:
    source_df = DATAFRAMES_BY_MP[menu_mp][discount_rate_short][rcm_model]
    for cs in REMDB_COST_SCENARIO_KEYS:
        for scenario_type, prefix in [('Pre-IRA', f'preIRA_mp{menu_mp}_'), ('IRA-Reference', f'iraRef_mp{menu_mp}_')]:
            col = create_adoption_col(
                prefix, category, 'adoption', cs, f'_{discount_rate_short}',
                scc_assumption=scc, rcm_model=rcm_model, cr_function=cr_function)
            if col in source_df.columns:
                mean_val = source_df[col].mean() * 100
                adoption_summary.append({
                    'MP': MP_LABELS[menu_mp],
                    'Cost Scenario': cs,
                    'Policy': scenario_type,
                    'Mean Adoption (%)': mean_val
                })

df_adoption_summary = pd.DataFrame(adoption_summary)

# Grouped bar chart
fig_cross, axes_cross = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for idx, policy in enumerate(['Pre-IRA', 'IRA-Reference']):
    ax = axes_cross[idx]
    df_policy = df_adoption_summary[df_adoption_summary['Policy'] == policy]

    if not df_policy.empty:
        pivot = df_policy.pivot(index='MP', columns='Cost Scenario', values='Mean Adoption (%)')
        # Reorder to match MP order
        mp_order = [MP_LABELS[mp] for mp in HEATING_MEASURE_PACKAGES if MP_LABELS[mp] in pivot.index]
        pivot = pivot.loc[mp_order]

        pivot.plot(kind='bar', ax=ax, width=0.7, edgecolor='black', linewidth=0.5)
        ax.set_title(f'{policy}', fontsize=13, fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel('Mean Adoption Rate (%)' if idx == 0 else '')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=10)
        ax.legend(title='Cost Scenario', fontsize=9)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=1))

        # Add value labels
        for container in ax.containers:
            ax.bar_label(container, fmt='%.1f%%', fontsize=7, padding=2)

fig_cross.suptitle(
    f'Mean Heating Adoption Rate by Retrofit Depth and Cost Scenario\n'
    f'(SCC={scc}, RCM={rcm_model}, CRF={cr_function}, DR={discount_rate_short})',
    fontsize=14, fontweight='bold', y=1.02
)
fig_cross.tight_layout()
plt.show()

# ─── 2. CAPITAL COST BOX PLOTS: MP × Cost Scenario ───
fig_cap, axes_cap = plt.subplots(1, len(HEATING_MEASURE_PACKAGES),
                                  figsize=(6 * len(HEATING_MEASURE_PACKAGES), 6),
                                  sharey=False)
if len(HEATING_MEASURE_PACKAGES) == 1:
    axes_cap = [axes_cap]

for idx, menu_mp in enumerate(HEATING_MEASURE_PACKAGES):
    ax = axes_cap[idx]
    source_df = DATAFRAMES_BY_MP[menu_mp][discount_rate_short][rcm_model]
    scenario_prefix_ira = f'iraRef_mp{menu_mp}_'

    cap_data = {}
    for cs in REMDB_COST_SCENARIO_KEYS:
        col = create_capital_col(scenario_prefix_ira, category, net=True, cost_scenario=cs)
        if col in source_df.columns:
            cap_data[cs] = source_df[col].dropna()

    if cap_data:
        bp = ax.boxplot(
            cap_data.values(),
            labels=cap_data.keys(),
            patch_artist=True,
            showfliers=False,
            medianprops=dict(color='black', linewidth=1.5)
        )
        colors = ['steelblue', 'coral', 'seagreen', 'goldenrod']
        for patch, color in zip(bp['boxes'], colors[:len(cap_data)]):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)

        ax.set_title(f'{MP_LABELS[menu_mp]}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Cost Scenario')
        ax.set_ylabel('Net Capital Cost ($)' if idx == 0 else '')
        ax.axhline(y=0, color='black', linestyle='--', linewidth=0.5, alpha=0.5)
        ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))

fig_cap.suptitle(
    'IRA-Reference Net Capital Cost Distribution by Retrofit Depth and Cost Scenario',
    fontsize=14, fontweight='bold', y=1.02
)
fig_cap.tight_layout()
plt.show()

# ─── 3. COMPREHENSIVE SUMMARY TABLE ───
print(f"\n{'='*100}")
print("COMPREHENSIVE CROSS-MP × COST SCENARIO SUMMARY")
print(f"{'='*100}")
print(f"\n{'MP':<18} {'Cost':<10} {'Adoption':>12} {'Adoption':>12} {'IRA':>10} "
      f"{'Median Net':>14} {'Median NPV':>14} {'% NPV>0':>10}")
print(f"{'':<18} {'Scen.':<10} {'Pre-IRA':>12} {'IRA-Ref':>12} {'Uplift':>10} "
      f"{'Cap. Cost':>14} {'(moreWTP)':>14} {'':>10}")
print(f"{'-'*100}")

for menu_mp in HEATING_MEASURE_PACKAGES:
    source_df = DATAFRAMES_BY_MP[menu_mp][discount_rate_short][rcm_model]
    scenario_prefix_pre = f'preIRA_mp{menu_mp}_'
    scenario_prefix_ira = f'iraRef_mp{menu_mp}_'

    for cs in REMDB_COST_SCENARIO_KEYS:
        # Adoption columns  
        col_pre = create_adoption_col(
            scenario_prefix_pre, category, 'adoption', cs, f'_{discount_rate_short}',
            scc_assumption=scc, rcm_model=rcm_model, cr_function=cr_function)
        col_ira = create_adoption_col(
            scenario_prefix_ira, category, 'adoption', cs, f'_{discount_rate_short}',
            scc_assumption=scc, rcm_model=rcm_model, cr_function=cr_function)

        # Capital cost column
        cap_col = create_capital_col(scenario_prefix_ira, category, net=True, cost_scenario=cs)

        # NPV column
        npv_col = create_npv_col(
            scenario_prefix_ira, category, 'moreWTP', cs, f'_{discount_rate_short}')

        # Calculate values
        pre_adopt = source_df[col_pre].mean() * 100 if col_pre in source_df.columns else float('nan')
        ira_adopt = source_df[col_ira].mean() * 100 if col_ira in source_df.columns else float('nan')
        uplift = ira_adopt - pre_adopt

        median_cap = source_df[cap_col].median() if cap_col in source_df.columns else float('nan')
        npv_data = source_df[npv_col].dropna() if npv_col in source_df.columns else pd.Series(dtype=float)
        median_npv = npv_data.median() if len(npv_data) > 0 else float('nan')
        pct_pos = (npv_data > 0).sum() / len(npv_data) * 100 if len(npv_data) > 0 else float('nan')

        print(f"{MP_LABELS[menu_mp]:<18} {cs:<10} {pre_adopt:>11.2f}% {ira_adopt:>11.2f}% "
              f"{uplift:>+9.2f}% {median_cap:>14,.0f} {median_npv:>14,.0f} {pct_pos:>9.1f}%")
    print()  # blank line between MPs

print(f"\nNote: All values use SCC={scc}, RCM={rcm_model}, CRF={cr_function}, DR={discount_rate_short}")
print(f"Capital costs are IRA-Reference scenario net of rebates.")

# Model Runtime

In [ ]:
# Get the current datetime again
end_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Calculate the elapsed time
elapsed_time = datetime.strptime(end_time, "%Y-%m-%d_%H-%M-%S") - datetime.strptime(start_time, "%Y-%m-%d_%H-%M-%S")

# Format the elapsed time
elapsed_seconds = elapsed_time.total_seconds()
elapsed_minutes = int(elapsed_seconds // 60)
elapsed_seconds = int(elapsed_seconds % 60)

# Print the elapsed time
print(f"The code took {elapsed_minutes} minutes and {elapsed_seconds} seconds to execute.")